# IDS-BGWO-SHAP — Smoke test (~60 seconds)

Fast end-to-end check that the pipeline wires together. Runs all 8 FS methods on a 3K-row
stratified sample with BGWO pop=3 / iter=2, single seed. Does **not** produce competitive
numbers — only proves the install, data loader, BGWO loop, SHAP fitness, LCCDE training,
metrics, and plot pipeline all work.

Uses the bundled CIC-IDS2017 sample CSV from Western-OC2-Lab's baseline repo. **No
`kaggle.json` upload required** — the CSV is fetched directly over HTTPS in ~5 seconds.

If this notebook finishes with an aggregate table and no traceback, push the
`notebooks/run_reduced.ipynb` button to launch the real (~3-hour) multi-seed run.

## 0. Reset the environment (run this first, then **Runtime → Restart session**, then **Run All**)

Colab's base image has a partially-updated numpy: `numpy.__version__` reports 2.0.2 but
`numpy/testing/_private/utils.py` references symbols only present in numpy ≥ 2.2's C
extension. Importing sklearn (which pulls in `numpy.testing` transitively) then
crashes with `AttributeError: module 'numpy._core._multiarray_umath' has no attribute
'_blas_supports_fpe'`.

This cell force-reinstalls numpy to a consistent 2.2+ ABI and removes any stale
`IDS-features-selection/` clone so cell 1 below pulls the latest `requirements.txt`.
After running it, restart the runtime so Python drops the broken in-memory numpy.

In [ ]:
import os, subprocess, sys, shutil, pathlib

# 1. Nuke the stale clone so the install cell pulls the fixed requirements.txt
if pathlib.Path('IDS-features-selection').exists():
    shutil.rmtree('IDS-features-selection')

# 2. Force-reinstall numpy so the in-runtime ABI matches what's on disk
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', '--no-deps', 'numpy>=2.2,<2.5'],
               check=True)

print('Cleaned. NOW: Runtime -> Restart session, then Run All.')

## 1. Clone the repo and install dependencies

In [ ]:
# Same install cell as run_colab.ipynb. Suppresses pip resolver noise
# and only prints a controlled status table.
import pathlib, subprocess, sys

REPO_DIR = 'IDS-features-selection'
if not pathlib.Path(REPO_DIR).exists():
    !git clone -q https://github.com/yazanjer/IDS-features-selection.git
%cd $REPO_DIR

print('Installing requirements (suppressing pip resolver chatter) ...')
proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check',
     '-r', 'requirements.txt'],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print('PIP INSTALL FAILED — full output below:')
    print(proc.stdout)
    print(proc.stderr)
    raise SystemExit(1)
print('Install OK.')

import importlib
for pkg in ('numpy', 'pandas', 'sklearn', 'scipy', 'lightgbm', 'xgboost',
            'shap', 'lime', 'imblearn', 'kagglehub'):
    m = importlib.import_module(pkg)
    print(f'  {pkg:12s} {getattr(m, "__version__", "?")}')
print('Environment ready.')

## 2. Fetch the bundled CIC-IDS2017 sample CSV

Pulls Western-OC2-Lab's baseline CSV (~19 MB). No Kaggle credentials needed.

In [ ]:
import pathlib, urllib.request, os

CSV_URL = (
    'https://github.com/Western-OC2-Lab/Intrusion-Detection-System-Using-Machine-Learning'
    '/raw/main/data/CICIDS2017_sample.csv'
)
CSV_PATH = pathlib.Path('/content/CICIDS2017_sample.csv')

if not CSV_PATH.exists():
    print(f'Downloading {CSV_URL} ...')
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
print(f'CSV ready: {CSV_PATH} ({CSV_PATH.stat().st_size/1e6:.1f} MB)')

## 3. Run the smoke matrix (all 8 methods × 1 seed, tiny budget)

Methods: `none`, `filter`, `rfe`, `lasso`, `rf_imp`, `boruta`, `bgwo_bi`, `bgwo_shap`.
Expected total runtime: 30–60 seconds depending on Colab load.

In [ ]:
from dataclasses import replace
from src.config import smoke_config
from src.evaluation import run_experiment_matrix, aggregate, DEFAULT_METHODS

# smoke_config(): 3K rows, pop=3, iter=2, 1 seed, SHAP bg=20, SMOTE min=200
# Point loader at the CSV we just downloaded.
cfg = replace(smoke_config(), local_cicids_csv=CSV_PATH)
cfg.ensure_dirs()
print(cfg.to_json())

df = run_experiment_matrix(cfg, methods=DEFAULT_METHODS)
agg = aggregate(df)
agg

## 4. Verify the pipeline produced sensible output

If the assertions below pass, every code path is healthy and you're safe to run the
longer notebook. **Numbers themselves are not meaningful at smoke budget** — don't read
anything into the per-method ranking here.

In [ ]:
expected = set(DEFAULT_METHODS)
got      = set(df['method'].unique())
missing  = expected - got
assert not missing, f'methods that failed to produce any rows: {missing}'

# Every trial should have selected at least 1 feature.
assert (df['n_features_selected'] > 0).all(), \
    'a trial selected zero features (empty-mask rescue failed?)'

# bgwo_shap should have produced a fitness history (= the BGWO inner loop ran).
bs = df[df['method'] == 'bgwo_shap'].iloc[0]
assert bs.get('n_features_selected', 0) > 0, 'bgwo_shap produced no subset'

print('Pipeline smoke PASS — ready for the reduced/full runs.')

In [ ]:
# Quick visual check: per-class F1 + Pareto plots for the smoke run.
from src.plots import plot_latency_vs_features, render_comparison_table
plot_latency_vs_features(df, cfg.results_dir)
render_comparison_table(agg, cfg.results_dir)

import pathlib
print('Plots written to:', cfg.results_dir)
for p in sorted(pathlib.Path(cfg.results_dir).glob('*.png'))[:10]:
    print(' ', p.name)